In [2]:
import joblib
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


In [ ]:
df = pd.read_csv('../data/loan_dataset_20000.csv')

In [3]:
df.columns.tolist()

['age',
 'gender',
 'marital_status',
 'education_level',
 'annual_income',
 'monthly_income',
 'employment_status',
 'debt_to_income_ratio',
 'credit_score',
 'loan_amount',
 'loan_purpose',
 'interest_rate',
 'loan_term',
 'installment',
 'grade_subgrade',
 'num_of_open_accounts',
 'total_credit_limit',
 'current_balance',
 'delinquency_history',
 'public_records',
 'num_of_delinquencies',
 'loan_paid_back']

In [4]:
df.head(1)

,age,gender,marital_status,education_level,annual_income,monthly_income,employment_status,debt_to_income_ratio,credit_score,loan_amount,...,loan_term,installment,grade_subgrade,num_of_open_accounts,total_credit_limit,current_balance,delinquency_history,public_records,num_of_delinquencies,loan_paid_back
0,59,Male,Married,Master's,24240.19,2020.02,Employed,0.074,743,17173.72,...,36,581.88,B5,7,40833.47,24302.07,1,0,1,1


In [5]:
df['loan_paid_back'].value_counts()

loan_paid_back
1    15998
0     4002
Name: count, dtype: int64

In [6]:
df['loan_paid_back'].value_counts(normalize=True).mul(100).round(2)

loan_paid_back
1    79.99
0    20.01
Name: proportion, dtype: float64

In [4]:

data = load_iris()
X = data.data
y = data.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

from sklearn.feature_selection import SequentialFeatureSelector

model = GaussianNB()

sfs = SequentialFeatureSelector(
    estimator=model,
    n_features_to_select=2,  # จำนวนฟีเจอร์ที่ต้องการเลือกไว้
    direction='backward',
    cv=5                     # ใช้ Cross-validation 5 folds
)
sfs.fit(X_train, y_train)

X_train_selected = sfs.transform(X_train)
X_test_selected = sfs.transform(X_test)

model.fit(X_train_selected, y_train)

# 4. ทำ Prediction
predictions = model.predict(X_test_selected)

accuracy = accuracy_score(y_test, predictions)

precision = precision_score(y_test, predictions, average='macro')
recall = recall_score(y_test, predictions, average='macro')
f1 = f1_score(y_test, predictions, average='macro')

metrics = {
    "accuracy": accuracy,
    "precision": precision,
    "recall": recall,
    "f1_score": f1,
    "TP": sum((predictions == 1) & (y_test == 1)),
    "TN": sum((predictions == 0) & (y_test == 0)),
    "FP": sum((predictions == 1) & (y_test == 0)),
    "FN": sum((predictions == 0) & (y_test == 1))
}

print(f"Model Metrics: {metrics}")


Model Metrics: {'accuracy': 1.0, 'precision': 1.0, 'recall': 1.0, 'f1_score': 1.0, 'TP': np.int64(13), 'TN': np.int64(19), 'FP': np.int64(0), 'FN': np.int64(0)}


In [ ]:

data = load_iris()
X = data.data
y = data.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

from sklearn.feature_selection import SequentialFeatureSelector

model = GaussianNB()

model.fit(X_train, y_train)

# 4. ทำ Prediction
predictions = model.predict(X_test)

accuracy = accuracy_score(y_test, predictions)

precision = precision_score(y_test, predictions, average='macro')
recall = recall_score(y_test, predictions, average='macro')
f1 = f1_score(y_test, predictions, average='macro')

metrics = {
    "accuracy": accuracy,
    "precision": precision,
    "recall": recall,
    "f1_score": f1,
    "TP": sum((predictions == 1) & (y_test == 1)),
    "TN": sum((predictions == 0) & (y_test == 0)),
    "FP": sum((predictions == 1) & (y_test == 0)),
    "FN": sum((predictions == 0) & (y_test == 1))
}

print(f"Model Metrics: {metrics}")


# All model version 1

In [1]:
# import os
# import pandas as pd
# import numpy as np
# from sklearn.feature_selection import SequentialFeatureSelector
# from sklearn.linear_model import LogisticRegression
# from sklearn.naive_bayes import GaussianNB
# from sklearn.ensemble import AdaBoostClassifier
# from sklearn.metrics import f1_score, precision_score
# from imblearn.over_sampling import SMOTE
# from imblearn.pipeline import Pipeline as ImbPipeline
# import warnings
# warnings.filterwarnings('ignore')

# # 1. กำหนด Models ตามเปเปอร์
# models = {
#     'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
#     'Naive Bayes': GaussianNB(),
#     'AdaBoost': AdaBoostClassifier(random_state=42)
# }

# # 2. ฟังก์ชันสำหรับเทรนและประเมินผลโดยอ่านจากโฟลเดอร์ Pre-split
# def evaluate_pre_split_folds(data_folder='data', use_feature_selection=False):
#     # เตรียม Dictionary เก็บผลลัพธ์
#     results = {name: {'f1': [], 'precision': []} for name in models.keys()}
    
#     # วนลูปตามโฟลเดอร์ fold1 ถึง fold5
#     for fold in range(1, 6):
#         print(f"\n--- Processing Fold {fold} ---")
        
#         # กำหนด Path ของไฟล์ Train และ Test (อ้างอิงจากโครงสร้างในภาพ)
#         train_file = os.path.join(data_folder, f'fold{fold}', f'fold_{fold}_train.csv')
#         test_file = os.path.join(data_folder, f'fold{fold}', f'fold_{fold}_test.csv')
        
#         # โหลดข้อมูล
#         train_df = pd.read_csv(train_file)
#         test_df = pd.read_csv(test_file)
        
#         # แยก Features (X) และ Target (y)
#         X_train = train_df.drop('loan_paid_back', axis=1)
#         y_train = train_df['loan_paid_back']
#         X_test = test_df.drop('loan_paid_back', axis=1)
#         y_test = test_df['loan_paid_back']
        
#         # เทรนและทดสอบทีละโมเดล
#         for name, model in models.items():
#             print(f"[{name}] Training...")
            
#             # สร้าง Pipeline
#             steps = [('smote', SMOTE(random_state=42))]
            
#             if use_feature_selection:
#                 sfs = SequentialFeatureSelector(
#                     model, 
#                     direction='backward', 
#                     scoring='f1', 
#                     cv=3, 
#                     n_jobs=-1
#                 )
#                 steps.append(('backward_selection', sfs))
                
#             steps.append(('classifier', model))
#             pipeline = ImbPipeline(steps)
            
#             # Fit กับข้อมูล Train ของรอบนี้
#             pipeline.fit(X_train, y_train)
            
#             # Predict กับข้อมูล Test ของรอบนี้
#             y_pred = pipeline.predict(X_test)
            
#             # คำนวณ Metrics และเก็บไว้
#             results[name]['f1'].append(f1_score(y_test, y_pred))
#             results[name]['precision'].append(precision_score(y_test, y_pred))

#     # คำนวณค่าเฉลี่ยของทั้ง 5 Folds
#     final_summary = {}
#     for name in models.keys():
#         final_summary[name] = {
#             'F1-Score': np.mean(results[name]['f1']),
#             'Precision': np.mean(results[name]['precision'])
#         }
        
#     return pd.DataFrame(final_summary).T

# # 3. รันการทดสอบ
# # หมายเหตุ: ตรวจสอบให้แน่ใจว่า รันโค้ดนี้ในระดับเดียวกับโฟลเดอร์ data
# print("=== Training on Full Feature Set ===")
# full_features_results = evaluate_pre_split_folds(data_folder='../data', use_feature_selection=False)

# print("\n=== Training on Selected Feature Subset (Backward Selection) ===")
# selected_features_results = evaluate_pre_split_folds(data_folder='../data', use_feature_selection=True)

# # 4. แสดงผลสรุป
# print("\n--- Results Summary (Full Features) ---")
# print(full_features_results)

# print("\n--- Results Summary (Selected Features) ---")
# print(selected_features_results)

In [3]:
import os
import json
import joblib
import pandas as pd
import numpy as np
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score, 
    precision_score, recall_score, confusion_matrix, 
    roc_auc_score, log_loss
)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import warnings
warnings.filterwarnings('ignore')

# 1. กำหนด Models ตามเปเปอร์
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Naive Bayes': GaussianNB(),
    'AdaBoost': AdaBoostClassifier(random_state=42)
}

# สร้างโฟลเดอร์สำหรับเก็บผลลัพธ์และโมเดล
os.makedirs('saved_models', exist_ok=True)
os.makedirs('saved_results', exist_ok=True)

# ฟังก์ชันคำนวณ Metrics ทั้งหมดตาม Format ที่ต้องการ
def calculate_metrics(y_true, y_pred, y_prob, prefix=""):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        f"{prefix}accuracy": accuracy_score(y_true, y_pred),
        f"{prefix}balanced-accuracy-score": balanced_accuracy_score(y_true, y_pred),
        f"{prefix}f1_score": f1_score(y_true, y_pred),
        f"{prefix}precision": precision_score(y_true, y_pred),
        f"{prefix}recall": recall_score(y_true, y_pred),
        f"{prefix}TN": int(tn),
        f"{prefix}FP": int(fp),
        f"{prefix}FN": int(fn),
        f"{prefix}TP": int(tp),
        f"{prefix}AUC": roc_auc_score(y_true, y_prob) if y_prob is not None else None,
        f"{prefix}loss-score": log_loss(y_true, y_prob) if y_prob is not None else None
    }

# 2. ฟังก์ชันสำหรับเทรนและประเมินผล
def evaluate_pre_split_folds(data_folder='data', use_feature_selection=False, exp_name="full_features"):
    # โครงสร้างเก็บผลลัพธ์
    final_results = {}
    
    for name, model in models.items():
        final_results[name] = {
            "selected_features_per_fold": {},
            "metrics_per_fold": {},
            "confusionMatrix": {} # สำหรับเก็บค่าเฉลี่ย
        }
        fold_metrics = []
        
        for fold in range(1, 6):
            print(f"\n--- Processing Fold {fold} | Model: {name} | Selection: {use_feature_selection} ---")
            
            # กำหนด Path ของไฟล์ Train และ Test
            train_file = os.path.join(data_folder, f'fold{fold}', f'fold_{fold}_train.csv')
            test_file = os.path.join(data_folder, f'fold{fold}', f'fold_{fold}_test.csv')
            
            train_df = pd.read_csv(train_file)
            test_df = pd.read_csv(test_file)
            
            X_train = train_df.drop('loan_paid_back', axis=1)
            y_train = train_df['loan_paid_back']
            X_test = test_df.drop('loan_paid_back', axis=1)
            y_test = test_df['loan_paid_back']
            
            # สร้าง Pipeline
            steps = [('smote', SMOTE(random_state=42))]
            
            if use_feature_selection:
                sfs = SequentialFeatureSelector(
                    model, 
                    direction='backward', 
                    scoring='f1', 
                    cv=3, 
                    n_jobs=-1
                )
                steps.append(('backward_selection', sfs))
                
            steps.append(('classifier', model))
            pipeline = ImbPipeline(steps)
            
            # Fit Model
            pipeline.fit(X_train, y_train)
            
            # บันทึกโมเดลด้วย joblib
            model_filename = f'saved_models/{exp_name}_{name.replace(" ", "_")}_fold{fold}.joblib'
            joblib.dump(pipeline, model_filename)
            
            # เก็บ Features ที่ถูกเลือก (ถ้ามีการทำ Feature Selection)
            if use_feature_selection:
                selected_mask = pipeline.named_steps['backward_selection'].get_support()
                selected_features = X_train.columns[selected_mask].tolist()
                final_results[name]["selected_features_per_fold"][f"fold_{fold}"] = selected_features
            else:
                final_results[name]["selected_features_per_fold"][f"fold_{fold}"] = X_train.columns.tolist()

            # Predict - Train data
            y_train_pred = pipeline.predict(X_train)
            y_train_prob = pipeline.predict_proba(X_train)[:, 1] if hasattr(pipeline, "predict_proba") else None
            
            # Predict - Test data
            y_test_pred = pipeline.predict(X_test)
            y_test_prob = pipeline.predict_proba(X_test)[:, 1] if hasattr(pipeline, "predict_proba") else None
            
            # คำนวณ Metrics
            train_metrics = calculate_metrics(y_train, y_train_pred, y_train_prob, prefix="train_")
            test_metrics = calculate_metrics(y_test, y_test_pred, y_test_prob, prefix="")
            
            # รวม Metrics ของ Fold นี้
            combined_metrics = {**train_metrics, **test_metrics}
            final_results[name]["metrics_per_fold"][f"fold_{fold}"] = combined_metrics
            fold_metrics.append(combined_metrics)
            
        # หาค่าเฉลี่ยของทั้ง 5 Folds เก็บใน "confusionMatrix" แบบที่กำหนด
        avg_metrics = {}
        for key in fold_metrics[0].keys():
            if fold_metrics[0][key] is not None:
                # ถ้าเป็น TN, FP, FN, TP ให้ปัดเป็นจำนวนเต็ม (เฉลี่ย)
                if key in ["train_TN", "train_FP", "train_FN", "train_TP", "TN", "FP", "FN", "TP"]:
                    avg_metrics[key] = int(np.mean([m[key] for m in fold_metrics]))
                else:
                    avg_metrics[key] = np.mean([m[key] for m in fold_metrics])
        
        final_results[name]["confusionMatrix"] = avg_metrics
        
    return final_results

# 3. รันการทดสอบและ Export เป็น JSON
print("=== Starting Experiment: Full Feature Set ===")
full_features_results = evaluate_pre_split_folds(data_folder='../data', use_feature_selection=False, exp_name="FullFeatures")

with open('saved_results/results_full_features.json', 'w', encoding='utf-8') as f:
    json.dump(full_features_results, f, ensure_ascii=False, indent=4)

print("\n=== Starting Experiment: Selected Feature Subset (Backward Selection) ===")
selected_features_results = evaluate_pre_split_folds(data_folder='../data', use_feature_selection=True, exp_name="SelectedFeatures")

with open('saved_results/results_selected_features.json', 'w', encoding='utf-8') as f:
    json.dump(selected_features_results, f, ensure_ascii=False, indent=4)

print("\nAll tasks completed! Models are saved in 'saved_models/' and JSON results are in 'saved_results/'.")

=== Starting Experiment: Full Feature Set ===

--- Processing Fold 1 | Model: Logistic Regression | Selection: False ---

--- Processing Fold 2 | Model: Logistic Regression | Selection: False ---

--- Processing Fold 3 | Model: Logistic Regression | Selection: False ---

--- Processing Fold 4 | Model: Logistic Regression | Selection: False ---

--- Processing Fold 5 | Model: Logistic Regression | Selection: False ---

--- Processing Fold 1 | Model: Naive Bayes | Selection: False ---

--- Processing Fold 2 | Model: Naive Bayes | Selection: False ---

--- Processing Fold 3 | Model: Naive Bayes | Selection: False ---

--- Processing Fold 4 | Model: Naive Bayes | Selection: False ---

--- Processing Fold 5 | Model: Naive Bayes | Selection: False ---

--- Processing Fold 1 | Model: AdaBoost | Selection: False ---

--- Processing Fold 2 | Model: AdaBoost | Selection: False ---

--- Processing Fold 3 | Model: AdaBoost | Selection: False ---

--- Processing Fold 4 | Model: AdaBoost | Selection

# All model version 2

In [ ]:
import os
import json
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt # สำหรับเซฟกราฟ
from sklearn.model_selection import cross_validate # สำหรับประเมินผลในแต่ละ Step
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score, 
    precision_score, recall_score, confusion_matrix, 
    roc_auc_score, log_loss
)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import warnings
warnings.filterwarnings('ignore')

# 1. กำหนด Models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Naive Bayes': GaussianNB(),
    'AdaBoost': AdaBoostClassifier(random_state=42)
}

# สร้างโฟลเดอร์สำหรับเก็บผลลัพธ์และโมเดล
os.makedirs('saved_models', exist_ok=True)
os.makedirs('saved_results', exist_ok=True)

# 2. ฟังก์ชันคำนวณ Metrics
def calculate_metrics(y_true, y_pred, y_prob, prefix=""):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        f"{prefix}accuracy": accuracy_score(y_true, y_pred),
        f"{prefix}balanced-accuracy-score": balanced_accuracy_score(y_true, y_pred),
        f"{prefix}f1_score": f1_score(y_true, y_pred),
        f"{prefix}precision": precision_score(y_true, y_pred),
        f"{prefix}recall": recall_score(y_true, y_pred),
        f"{prefix}TN": int(tn),
        f"{prefix}FP": int(fp),
        f"{prefix}FN": int(fn),
        f"{prefix}TP": int(tp),
        f"{prefix}AUC": roc_auc_score(y_true, y_prob) if y_prob is not None else None,
        f"{prefix}loss-score": log_loss(y_true, y_prob) if y_prob is not None else None
    }

# 3. ฟังก์ชัน Custom Greedy Backward Selection พร้อมเก็บประวัติ
def greedy_backward_selection_with_tracking(model, X, y, cv=3, min_features=1):
    current_features = list(X.columns)
    history = []
    
    # คำนวณ Baseline (ก่อนตัดฟีเจอร์)
    pipeline = ImbPipeline([('smote', SMOTE(random_state=42)), ('classifier', model)])
    scores = cross_validate(pipeline, X, y, cv=cv, scoring=['f1', 'precision', 'roc_auc'], n_jobs=-1)
    
    history.append({
        'Step': 0,
        'Removed_Feature': 'None (Baseline)',
        'Remaining_Features_Count': len(current_features),
        'AUC': np.mean(scores['test_roc_auc']),
        'Precision': np.mean(scores['test_precision']),
        'F1': np.mean(scores['test_f1']),
        'Remaining_Features': list(current_features)
    })
    
    # วนลูปตัดฟีเจอร์ออกทีละ 1 ตัว จนกว่าจะเหลือเท่ากับ min_features
    while len(current_features) > min_features:
        best_f1 = -1
        worst_feature = None
        best_metrics = {}
        
        # ทดสอบตัดฟีเจอร์ออกทีละตัวเพื่อหาว่าตัวไหนถูกตัดแล้วผลลัพธ์ดีที่สุด
        for feature in current_features:
            features_to_try = [f for f in current_features if f != feature]
            
            pipeline = ImbPipeline([('smote', SMOTE(random_state=42)), ('classifier', model)])
            scores = cross_validate(
                pipeline, X[features_to_try], y, 
                cv=cv, scoring=['f1', 'precision', 'roc_auc'], n_jobs=-1
            )
            
            mean_f1 = np.mean(scores['test_f1'])
            if mean_f1 > best_f1:
                best_f1 = mean_f1
                worst_feature = feature
                best_metrics = {
                    'F1': mean_f1, 
                    'Precision': np.mean(scores['test_precision']), 
                    'AUC': np.mean(scores['test_roc_auc'])
                }
                
        # นำฟีเจอร์ที่แย่ที่สุด (ตัดแล้ว F1 ยังสูง) ออกจากกลุ่ม
        current_features.remove(worst_feature)
        
        # บันทึกประวัติรอบนี้
        history.append({
            'Step': history[-1]['Step'] + 1,
            'Removed_Feature': worst_feature,
            'Remaining_Features_Count': len(current_features),
            'AUC': best_metrics['AUC'],
            'Precision': best_metrics['Precision'],
            'F1': best_metrics['F1'],
            'Remaining_Features': list(current_features)
        })
        print(f"   [Step {history[-1]['Step']}] Removed: {worst_feature} | Current F1: {best_metrics['F1']:.4f}")
        
    return pd.DataFrame(history)

# 4. ฟังก์ชันหลักสำหรับเทรน
def evaluate_pre_split_folds(data_folder='../data', use_feature_selection=False, exp_name="full_features"):
    final_results = {}
    
    for name, model in models.items():
        final_results[name] = {
            "selected_features_per_fold": {},
            "metrics_per_fold": {},
            "confusionMatrix": {}
        }
        fold_metrics = []
        
        for fold in range(1, 6):
            print(f"\n--- Processing Fold {fold} | Model: {name} | Selection: {use_feature_selection} ---")
            
            train_file = os.path.join(data_folder, f'fold{fold}', f'fold_{fold}_train.csv')
            test_file = os.path.join(data_folder, f'fold{fold}', f'fold_{fold}_test.csv')
            
            train_df = pd.read_csv(train_file)
            test_df = pd.read_csv(test_file)
            
            X_train = train_df.drop('loan_paid_back', axis=1)
            y_train = train_df['loan_paid_back']
            X_test = test_df.drop('loan_paid_back', axis=1)
            y_test = test_df['loan_paid_back']
            
            if use_feature_selection:
                # รันกระบวนการ Backward Selection เพื่อหาประวัติและเซฟ
                history_df = greedy_backward_selection_with_tracking(model, X_train, y_train, cv=3, min_features=1)
                
                # ก. บันทึกเป็น CSV (แปลง List กลับเป็นข้อความยาวๆ เพื่อให้เซฟลงตารางได้สวยงาม)
                csv_df = history_df.copy()
                csv_df['Remaining_Features'] = csv_df['Remaining_Features'].apply(lambda x: ", ".join(x))
                csv_path = f'saved_results/{exp_name}_{name.replace(" ", "_")}_fold{fold}_selection_history.csv'
                csv_df.to_csv(csv_path, index=False)
                
                # ข. สร้างและเซฟกราฟ AUC, Precision, F1 ในแต่ละ Step
                plt.figure(figsize=(10, 6))
                plt.plot(history_df['Step'], history_df['AUC'], marker='o', label='AUC')
                plt.plot(history_df['Step'], history_df['F1'], marker='s', label='F1')
                plt.plot(history_df['Step'], history_df['Precision'], marker='^', label='Precision')
                plt.xlabel('Number of Features Removed')
                plt.ylabel('Score')
                plt.title(f'Backward Selection History - {name} (Fold {fold})')
                plt.legend()
                plt.grid(True)
                plt.savefig(f'saved_results/{exp_name}_{name.replace(" ", "_")}_fold{fold}_selection_plot.png')
                plt.close()
                
                # ค. หาจุดตัดที่ดีที่สุด (อิงจาก F1 สูงสุด) เพื่อนำฟีเจอร์มาใช้งาน
                optimal_step = history_df.loc[history_df['F1'].idxmax()]
                selected_features = optimal_step['Remaining_Features']
                final_results[name]["selected_features_per_fold"][f"fold_{fold}"] = selected_features
                
                # เลือกเฉพาะคอลัมน์ที่ผ่านการคัดเลือกมาเทรน Pipeline จริง
                X_train_fold = X_train[selected_features]
                X_test_fold = X_test[selected_features]
            else:
                # กรณี Full Features ใช้ทุกคอลัมน์
                X_train_fold = X_train
                X_test_fold = X_test
                final_results[name]["selected_features_per_fold"][f"fold_{fold}"] = list(X_train.columns)

            # เทรนและเก็บโมเดล
            pipeline = ImbPipeline([('smote', SMOTE(random_state=42)), ('classifier', model)])
            pipeline.fit(X_train_fold, y_train)
            
            model_filename = f'saved_models/{exp_name}_{name.replace(" ", "_")}_fold{fold}.joblib'
            joblib.dump(pipeline, model_filename)
            
            # Predict
            y_train_pred = pipeline.predict(X_train_fold)
            y_train_prob = pipeline.predict_proba(X_train_fold)[:, 1] if hasattr(pipeline, "predict_proba") else None
            y_test_pred = pipeline.predict(X_test_fold)
            y_test_prob = pipeline.predict_proba(X_test_fold)[:, 1] if hasattr(pipeline, "predict_proba") else None
            
            # เก็บผล Metrics
            train_metrics = calculate_metrics(y_train, y_train_pred, y_train_prob, prefix="train_")
            test_metrics = calculate_metrics(y_test, y_test_pred, y_test_prob, prefix="")
            
            combined_metrics = {**train_metrics, **test_metrics}
            final_results[name]["metrics_per_fold"][f"fold_{fold}"] = combined_metrics
            fold_metrics.append(combined_metrics)
            
        # รวมค่าเฉลี่ย Metrics
        avg_metrics = {}
        for key in fold_metrics[0].keys():
            if fold_metrics[0][key] is not None:
                if key in ["train_TN", "train_FP", "train_FN", "train_TP", "TN", "FP", "FN", "TP"]:
                    avg_metrics[key] = int(np.mean([m[key] for m in fold_metrics]))
                else:
                    avg_metrics[key] = np.mean([m[key] for m in fold_metrics])
        
        final_results[name]["confusionMatrix"] = avg_metrics
        
    return final_results

# 5. สั่งรัน
print("=== Starting Experiment: Full Feature Set ===")
full_features_results = evaluate_pre_split_folds(data_folder='../data', use_feature_selection=False, exp_name="FullFeatures")
with open('saved_results/results_full_features.json', 'w', encoding='utf-8') as f:
    json.dump(full_features_results, f, ensure_ascii=False, indent=4)

print("\n=== Starting Experiment: Selected Feature Subset (Backward Selection) ===")
selected_features_results = evaluate_pre_split_folds(data_folder='../data', use_feature_selection=True, exp_name="SelectedFeatures")
with open('saved_results/results_selected_features.json', 'w', encoding='utf-8') as f:
    json.dump(selected_features_results, f, ensure_ascii=False, indent=4)

print("\nAll tasks completed! Check 'saved_results/' for CSV tables, Plots, and JSON metrics.")

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# กำหนดชื่อโมเดลและ Metric ที่ต้องการพลอตรวม
model_names = ['Logistic_Regression', 'Naive_Bayes', 'AdaBoost']
exp_name = "SelectedFeatures"
metric_to_plot = "F1"  # สามารถเปลี่ยนเป็น 'AUC' หรือ 'Precision' ได้

for model in model_names:
    plt.figure(figsize=(12, 7))
    
    # วนดึงข้อมูล CSV ของทั้ง 5 Fold มาวาดเส้นในกราฟเดียวกัน
    for fold in range(1, 6):
        csv_path = f'saved_results/{exp_name}_{model}_fold{fold}_selection_history.csv'
        
        if os.path.exists(csv_path):
            df = pd.read_csv(csv_path)
            plt.plot(df['Step'], df[metric_to_plot], marker='o', label=f'Fold {fold}')
            
    plt.xlabel('Number of Features Removed')
    plt.ylabel(metric_to_plot)
    plt.title(f'Backward Selection ({metric_to_plot}) - {model.replace("_", " ")} - All Folds')
    plt.legend()
    plt.grid(True)
    
    # เซฟกราฟรวม
    save_path = f'saved_results/{exp_name}_{model}_Combined_{metric_to_plot}_plot.png'
    plt.savefig(save_path)
    plt.close()
    
    print(f"Saved combined plot: {save_path}")

# Version 3

## F1 

In [4]:
import os
import json
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt # สำหรับเซฟกราฟ
from sklearn.model_selection import cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score, 
    precision_score, recall_score, confusion_matrix, 
    roc_auc_score, log_loss
)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import warnings
warnings.filterwarnings('ignore')

# 1. กำหนด Models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Naive Bayes': GaussianNB(),
    'AdaBoost': AdaBoostClassifier(random_state=42)
}

# สร้างโฟลเดอร์หลัก
os.makedirs('saved_models', exist_ok=True)
os.makedirs('saved_results', exist_ok=True)

# 2. ฟังก์ชันคำนวณ Metrics
def calculate_metrics(y_true, y_pred, y_prob, prefix=""):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        f"{prefix}accuracy": accuracy_score(y_true, y_pred),
        f"{prefix}balanced-accuracy-score": balanced_accuracy_score(y_true, y_pred),
        f"{prefix}f1_score": f1_score(y_true, y_pred),
        f"{prefix}precision": precision_score(y_true, y_pred),
        f"{prefix}recall": recall_score(y_true, y_pred),
        f"{prefix}TN": int(tn),
        f"{prefix}FP": int(fp),
        f"{prefix}FN": int(fn),
        f"{prefix}TP": int(tp),
        f"{prefix}AUC": roc_auc_score(y_true, y_prob) if y_prob is not None else None,
        f"{prefix}loss-score": log_loss(y_true, y_prob) if y_prob is not None else None
    }

# 3. ฟังก์ชัน Custom Greedy Backward Selection พร้อมเก็บประวัติ
def greedy_backward_selection_with_tracking(model, X, y, cv=3, min_features=1):
    current_features = list(X.columns)
    history = []
    
    # คำนวณ Baseline (ก่อนตัดฟีเจอร์)
    pipeline = ImbPipeline([('smote', SMOTE(random_state=42)), ('classifier', model)])
    scores = cross_validate(pipeline, X, y, cv=cv, scoring=['f1', 'precision', 'roc_auc'], n_jobs=-1)
    
    history.append({
        'Step': 0,
        'Removed_Feature': 'None (Baseline)',
        'Remaining_Features_Count': len(current_features),
        'AUC': np.mean(scores['test_roc_auc']),
        'Precision': np.mean(scores['test_precision']),
        'F1': np.mean(scores['test_f1']),
        'Remaining_Features': list(current_features)
    })
    
    # วนลูปตัดฟีเจอร์ออกทีละ 1 ตัว
    while len(current_features) > min_features:
        best_f1 = -1
        worst_feature = None
        best_metrics = {}
        
        for feature in current_features:
            features_to_try = [f for f in current_features if f != feature]
            
            pipeline = ImbPipeline([('smote', SMOTE(random_state=42)), ('classifier', model)])
            scores = cross_validate(
                pipeline, X[features_to_try], y, 
                cv=cv, scoring=['f1', 'precision', 'roc_auc'], n_jobs=-1
            )
            
            mean_f1 = np.mean(scores['test_f1'])
            if mean_f1 > best_f1:
                best_f1 = mean_f1
                worst_feature = feature
                best_metrics = {
                    'F1': mean_f1, 
                    'Precision': np.mean(scores['test_precision']), 
                    'AUC': np.mean(scores['test_roc_auc'])
                }
                
        current_features.remove(worst_feature)
        
        history.append({
            'Step': history[-1]['Step'] + 1,
            'Removed_Feature': worst_feature,
            'Remaining_Features_Count': len(current_features),
            'AUC': best_metrics['AUC'],
            'Precision': best_metrics['Precision'],
            'F1': best_metrics['F1'],
            'Remaining_Features': list(current_features)
        })
        print(f"   [Step {history[-1]['Step']}] Removed: {worst_feature} | Current F1: {best_metrics['F1']:.4f}")
        
    return pd.DataFrame(history)

# 4. ฟังก์ชันหลักสำหรับเทรน
def evaluate_pre_split_folds(data_folder='../data', use_feature_selection=False, exp_name="full_features"):
    final_results = {}
    
    for name, model in models.items():
        # สร้าง Sub-folder สำหรับแต่ละโมเดล
        safe_model_name = name.replace(" ", "_")
        model_save_dir = os.path.join('saved_models', safe_model_name)
        result_save_dir = os.path.join('saved_results', safe_model_name)
        
        os.makedirs(model_save_dir, exist_ok=True)
        os.makedirs(result_save_dir, exist_ok=True)

        final_results[name] = {
            "selected_features_per_fold": {},
            "metrics_per_fold": {},
            "confusionMatrix": {}
        }
        fold_metrics = []
        model_selection_histories = {} 
        
        for fold in range(1, 6):
            print(f"\n--- Processing Fold {fold} | Model: {name} | Selection: {use_feature_selection} ---")
            
            train_file = os.path.join(data_folder, f'fold{fold}', f'fold_{fold}_train.csv')
            test_file = os.path.join(data_folder, f'fold{fold}', f'fold_{fold}_test.csv')
            
            train_df = pd.read_csv(train_file)
            test_df = pd.read_csv(test_file)
            
            X_train = train_df.drop('loan_paid_back', axis=1)
            y_train = train_df['loan_paid_back']
            X_test = test_df.drop('loan_paid_back', axis=1)
            y_test = test_df['loan_paid_back']
            
            if use_feature_selection:
                history_df = greedy_backward_selection_with_tracking(model, X_train, y_train, cv=3, min_features=1)
                
                # ก. เซฟ CSV แต่ละ Fold
                csv_df = history_df.copy()
                csv_df['Remaining_Features'] = csv_df['Remaining_Features'].apply(lambda x: ", ".join(x))
                csv_path = os.path.join(result_save_dir, f'{exp_name}_fold{fold}_selection_history.csv')
                csv_df.to_csv(csv_path, index=False)
                
                # ข. สร้างและเซฟกราฟแยกทีละ Fold (Individual plot)
                plt.figure(figsize=(10, 6))
                plt.plot(history_df['Step'], history_df['AUC'], marker='o', label='AUC')
                plt.plot(history_df['Step'], history_df['F1'], marker='s', label='F1')
                plt.plot(history_df['Step'], history_df['Precision'], marker='^', label='Precision')
                plt.xlabel('Number of Features Removed')
                plt.ylabel('Score')
                plt.title(f'Backward Selection History - {name} (Fold {fold})')
                plt.legend()
                plt.grid(True)
                plt.ylim([0.0, 1.05])
                plot_indiv_path = os.path.join(result_save_dir, f'{exp_name}_fold{fold}_selection_plot.png')
                plt.savefig(plot_indiv_path)
                plt.close()
                
                model_selection_histories[fold] = history_df
                
                # หาจุดตัดฟีเจอร์ที่ดีที่สุด
                optimal_step = history_df.loc[history_df['F1'].idxmax()]
                selected_features = optimal_step['Remaining_Features']
                final_results[name]["selected_features_per_fold"][f"fold_{fold}"] = selected_features
                
                X_train_fold = X_train[selected_features]
                X_test_fold = X_test[selected_features]
            else:
                X_train_fold = X_train
                X_test_fold = X_test
                final_results[name]["selected_features_per_fold"][f"fold_{fold}"] = list(X_train.columns)

            # เทรนและเก็บโมเดล .joblib
            pipeline = ImbPipeline([('smote', SMOTE(random_state=42)), ('classifier', model)])
            pipeline.fit(X_train_fold, y_train)
            
            model_filename = os.path.join(model_save_dir, f'{exp_name}_fold{fold}.joblib')
            joblib.dump(pipeline, model_filename)
            
            # Predict
            y_train_pred = pipeline.predict(X_train_fold)
            y_train_prob = pipeline.predict_proba(X_train_fold)[:, 1] if hasattr(pipeline, "predict_proba") else None
            y_test_pred = pipeline.predict(X_test_fold)
            y_test_prob = pipeline.predict_proba(X_test_fold)[:, 1] if hasattr(pipeline, "predict_proba") else None
            
            # เก็บผล Metrics
            train_metrics = calculate_metrics(y_train, y_train_pred, y_train_prob, prefix="train_")
            test_metrics = calculate_metrics(y_test, y_test_pred, y_test_prob, prefix="")
            
            combined_metrics = {**train_metrics, **test_metrics}
            final_results[name]["metrics_per_fold"][f"fold_{fold}"] = combined_metrics
            fold_metrics.append(combined_metrics)
            
        # สร้างกราฟรวม (หลังรันครบ 5 Fold)
        if use_feature_selection and len(model_selection_histories) == 5:
            # 1. กราฟ Subplot รวมทุก Metric 5 ช่อง
            fig, axes = plt.subplots(1, 5, figsize=(25, 5), sharey=True)
            fig.suptitle(f'Backward Selection History - {name}', fontsize=16)
            
            for fold, ax in zip(range(1, 6), axes):
                h_df = model_selection_histories[fold]
                ax.plot(h_df['Step'], h_df['AUC'], marker='o', label='AUC')
                ax.plot(h_df['Step'], h_df['F1'], marker='s', label='F1')
                ax.plot(h_df['Step'], h_df['Precision'], marker='^', label='Precision')
                
                ax.set_title(f'Fold {fold}')
                ax.set_xlabel('Number of Features Removed')
                ax.grid(True)
                ax.set_ylim([0.0, 1.05])
                
            axes[0].set_ylabel('Score')
            axes[-1].legend(loc='lower left', bbox_to_anchor=(1.05, 0.5))
            
            plt.tight_layout()
            plot_combined_path = os.path.join(result_save_dir, f'{exp_name}_combined_metrics_plot.png')
            plt.savefig(plot_combined_path, bbox_inches='tight')
            plt.close()
            
            # 2. กราฟเปรียบเทียบเส้น AUC ของทุก Fold (แกนเดียวกัน)
            plt.figure(figsize=(10, 6))
            for fold in range(1, 6):
                h_df = model_selection_histories[fold]
                plt.plot(h_df['Step'], h_df['AUC'], marker='o', label=f'Fold {fold} AUC')
                
            plt.title(f'AUC Comparison Across All 5 Folds - {name}')
            plt.xlabel('Number of Features Removed')
            plt.ylabel('AUC Score')
            plt.grid(True)
            plt.ylim([0.0, 1.05])
            plt.legend()
            
            plot_auc_compare_path = os.path.join(result_save_dir, f'{exp_name}_AUC_comparison_all_folds.png')
            plt.savefig(plot_auc_compare_path)
            plt.close()

        # รวมค่าเฉลี่ย Metrics
        avg_metrics = {}
        for key in fold_metrics[0].keys():
            if fold_metrics[0][key] is not None:
                if key in ["train_TN", "train_FP", "train_FN", "train_TP", "TN", "FP", "FN", "TP"]:
                    avg_metrics[key] = int(np.mean([m[key] for m in fold_metrics]))
                else:
                    avg_metrics[key] = np.mean([m[key] for m in fold_metrics])
        
        final_results[name]["confusionMatrix"] = avg_metrics
        
    return final_results

# 5. สั่งรัน
print("=== Starting Experiment: Full Feature Set ===")
full_features_results = evaluate_pre_split_folds(data_folder='../data', use_feature_selection=False, exp_name="FullFeatures")

with open('saved_results/results_full_features.json', 'w', encoding='utf-8') as f:
    json.dump(full_features_results, f, ensure_ascii=False, indent=4)

print("\n=== Starting Experiment: Selected Feature Subset (Backward Selection) ===")
selected_features_results = evaluate_pre_split_folds(data_folder='../data', use_feature_selection=True, exp_name="SelectedFeatures")

with open('saved_results/results_selected_features.json', 'w', encoding='utf-8') as f:
    json.dump(selected_features_results, f, ensure_ascii=False, indent=4)

print("\nAll tasks completed! Check 'saved_results/' and 'saved_models/' for categorized files.")

=== Starting Experiment: Full Feature Set ===

--- Processing Fold 1 | Model: Logistic Regression | Selection: False ---

--- Processing Fold 2 | Model: Logistic Regression | Selection: False ---

--- Processing Fold 3 | Model: Logistic Regression | Selection: False ---

--- Processing Fold 4 | Model: Logistic Regression | Selection: False ---

--- Processing Fold 5 | Model: Logistic Regression | Selection: False ---

--- Processing Fold 1 | Model: Naive Bayes | Selection: False ---

--- Processing Fold 2 | Model: Naive Bayes | Selection: False ---

--- Processing Fold 3 | Model: Naive Bayes | Selection: False ---

--- Processing Fold 4 | Model: Naive Bayes | Selection: False ---

--- Processing Fold 5 | Model: Naive Bayes | Selection: False ---

--- Processing Fold 1 | Model: AdaBoost | Selection: False ---

--- Processing Fold 2 | Model: AdaBoost | Selection: False ---

--- Processing Fold 3 | Model: AdaBoost | Selection: False ---

--- Processing Fold 4 | Model: AdaBoost | Selection

## AUC

In [1]:
import os
import json
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score, 
    precision_score, recall_score, confusion_matrix, 
    roc_auc_score, log_loss
)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import warnings
warnings.filterwarnings('ignore')

# 1. กำหนด Models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Naive Bayes': GaussianNB(),
    'AdaBoost': AdaBoostClassifier(random_state=42)
}

# สร้างโฟลเดอร์หลัก
os.makedirs('saved_models', exist_ok=True)
os.makedirs('saved_results', exist_ok=True)

# 2. ฟังก์ชันคำนวณ Metrics
def calculate_metrics(y_true, y_pred, y_prob, prefix=""):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        f"{prefix}accuracy": accuracy_score(y_true, y_pred),
        f"{prefix}balanced-accuracy-score": balanced_accuracy_score(y_true, y_pred),
        f"{prefix}f1_score": f1_score(y_true, y_pred),
        f"{prefix}precision": precision_score(y_true, y_pred),
        f"{prefix}recall": recall_score(y_true, y_pred),
        f"{prefix}TN": int(tn),
        f"{prefix}FP": int(fp),
        f"{prefix}FN": int(fn),
        f"{prefix}TP": int(tp),
        f"{prefix}AUC": roc_auc_score(y_true, y_prob) if y_prob is not None else None,
        f"{prefix}loss-score": log_loss(y_true, y_prob) if y_prob is not None else None
    }

# 3.1 ฟังก์ชัน Custom Greedy Backward Selection (ตัดสินใจด้วย AUC)
def greedy_backward_selection_with_tracking(model, X, y, cv=3, min_features=1):
    current_features = list(X.columns)
    history = []
    
    pipeline = ImbPipeline([('smote', SMOTE(random_state=42)), ('classifier', model)])
    scores = cross_validate(pipeline, X, y, cv=cv, scoring=['f1', 'precision', 'roc_auc'], n_jobs=-1)
    
    history.append({
        'Step': 0,
        'Feature_Changed': 'None (Baseline)',
        'Remaining_Features_Count': len(current_features),
        'AUC': np.mean(scores['test_roc_auc']),
        'Precision': np.mean(scores['test_precision']),
        'F1': np.mean(scores['test_f1']),
        'Selected_Features': list(current_features)
    })
    
    while len(current_features) > min_features:
        best_auc = -1
        worst_feature = None
        best_metrics = {}
        
        for feature in current_features:
            features_to_try = [f for f in current_features if f != feature]
            
            pipeline = ImbPipeline([('smote', SMOTE(random_state=42)), ('classifier', model)])
            scores = cross_validate(
                pipeline, X[features_to_try], y, 
                cv=cv, scoring=['f1', 'precision', 'roc_auc'], n_jobs=-1
            )
            
            mean_auc = np.mean(scores['test_roc_auc'])
            
            if mean_auc > best_auc:
                best_auc = mean_auc
                worst_feature = feature
                best_metrics = {
                    'F1': np.mean(scores['test_f1']), 
                    'Precision': np.mean(scores['test_precision']), 
                    'AUC': mean_auc
                }
                
        current_features.remove(worst_feature)
        
        history.append({
            'Step': history[-1]['Step'] + 1,
            'Feature_Changed': f"- {worst_feature}", # เครื่องหมายลบ (เอาออก)
            'Remaining_Features_Count': len(current_features),
            'AUC': best_metrics['AUC'],
            'Precision': best_metrics['Precision'],
            'F1': best_metrics['F1'],
            'Selected_Features': list(current_features)
        })
        print(f"   [Backward Step {history[-1]['Step']}] Removed: {worst_feature} | Current AUC: {best_metrics['AUC']:.4f}")
        
    return pd.DataFrame(history)

# 3.2 ฟังก์ชัน Custom Greedy Forward Selection (ตัดสินใจด้วย AUC)
def greedy_forward_selection_with_tracking(model, X, y, cv=3):
    available_features = list(X.columns)
    current_features = []
    history = []
    step = 1
    
    while len(available_features) > 0:
        best_auc = -1
        best_feature_to_add = None
        best_metrics = {}
        
        for feature in available_features:
            features_to_try = current_features + [feature] # ลองเพิ่ม 1 ฟีเจอร์
            
            pipeline = ImbPipeline([('smote', SMOTE(random_state=42)), ('classifier', model)])
            scores = cross_validate(
                pipeline, X[features_to_try], y, 
                cv=cv, scoring=['f1', 'precision', 'roc_auc'], n_jobs=-1
            )
            
            mean_auc = np.mean(scores['test_roc_auc'])
            
            if mean_auc > best_auc:
                best_auc = mean_auc
                best_feature_to_add = feature
                best_metrics = {
                    'F1': np.mean(scores['test_f1']), 
                    'Precision': np.mean(scores['test_precision']), 
                    'AUC': mean_auc
                }
                
        # นำฟีเจอร์ที่ดีที่สุดในรอบนี้เก็บไว้
        current_features.append(best_feature_to_add)
        available_features.remove(best_feature_to_add)
        
        history.append({
            'Step': step,
            'Feature_Changed': f"+ {best_feature_to_add}", # เครื่องหมายบวก (เพิ่มเข้า)
            'Remaining_Features_Count': len(current_features),
            'AUC': best_metrics['AUC'],
            'Precision': best_metrics['Precision'],
            'F1': best_metrics['F1'],
            'Selected_Features': list(current_features)
        })
        print(f"   [Forward Step {step}] Added: {best_feature_to_add} | Current AUC: {best_metrics['AUC']:.4f}")
        step += 1
        
    return pd.DataFrame(history)

# 4. ฟังก์ชันหลักสำหรับเทรน (รองรับ selection_method = None, 'backward', 'forward')
def evaluate_pre_split_folds(data_folder='../data', selection_method=None, exp_name="full_features"):
    final_results = {}
    
    for name, model in models.items():
        safe_model_name = name.replace(" ", "_")
        model_save_dir = os.path.join('saved_models', safe_model_name)
        result_save_dir = os.path.join('saved_results', safe_model_name)
        
        os.makedirs(model_save_dir, exist_ok=True)
        os.makedirs(result_save_dir, exist_ok=True)

        final_results[name] = {
            "selected_features_per_fold": {},
            "metrics_per_fold": {},
            "confusionMatrix": {}
        }
        fold_metrics = []
        model_selection_histories = {} 
        
        for fold in range(1, 6):
            print(f"\n--- Processing Fold {fold} | Model: {name} | Method: {selection_method} ---")
            
            train_file = os.path.join(data_folder, f'fold{fold}', f'fold_{fold}_train.csv')
            test_file = os.path.join(data_folder, f'fold{fold}', f'fold_{fold}_test.csv')
            
            train_df = pd.read_csv(train_file)
            test_df = pd.read_csv(test_file)
            
            X_train = train_df.drop('loan_paid_back', axis=1)
            y_train = train_df['loan_paid_back']
            X_test = test_df.drop('loan_paid_back', axis=1)
            y_test = test_df['loan_paid_back']
            
            if selection_method == 'backward':
                history_df = greedy_backward_selection_with_tracking(model, X_train, y_train, cv=3, min_features=1)
            elif selection_method == 'forward':
                history_df = greedy_forward_selection_with_tracking(model, X_train, y_train, cv=3)
            else:
                history_df = None

            if history_df is not None:
                # ก. เซฟ CSV
                csv_df = history_df.copy()
                csv_df['Selected_Features'] = csv_df['Selected_Features'].apply(lambda x: ", ".join(x))
                csv_path = os.path.join(result_save_dir, f'{exp_name}_fold{fold}_selection_history.csv')
                csv_df.to_csv(csv_path, index=False)
                
                # ข. เซฟกราฟราย Fold
                x_label = 'Number of Features Removed' if selection_method == 'backward' else 'Number of Features Added'
                plt.figure(figsize=(10, 6))
                plt.plot(history_df['Step'], history_df['AUC'], marker='o', label='AUC')
                plt.plot(history_df['Step'], history_df['F1'], marker='s', label='F1')
                plt.plot(history_df['Step'], history_df['Precision'], marker='^', label='Precision')
                plt.xlabel(x_label)
                plt.ylabel('Score')
                plt.title(f'{selection_method.capitalize()} Selection - {name} (Fold {fold})')
                plt.legend()
                plt.grid(True)
                plt.ylim([0.0, 1.05])
                plot_indiv_path = os.path.join(result_save_dir, f'{exp_name}_fold{fold}_selection_plot.png')
                plt.savefig(plot_indiv_path)
                plt.close()
                
                model_selection_histories[fold] = history_df
                
                # เลือกจุดที่ดีที่สุดจาก AUC 
                optimal_step = history_df.loc[history_df['AUC'].idxmax()]
                selected_features = optimal_step['Selected_Features']
                final_results[name]["selected_features_per_fold"][f"fold_{fold}"] = selected_features
                
                X_train_fold = X_train[selected_features]
                X_test_fold = X_test[selected_features]
            else:
                X_train_fold = X_train
                X_test_fold = X_test
                final_results[name]["selected_features_per_fold"][f"fold_{fold}"] = list(X_train.columns)

            # เทรน
            pipeline = ImbPipeline([('smote', SMOTE(random_state=42)), ('classifier', model)])
            pipeline.fit(X_train_fold, y_train)
            
            model_filename = os.path.join(model_save_dir, f'{exp_name}_fold{fold}.joblib')
            joblib.dump(pipeline, model_filename)
            
            # Predict
            y_train_pred = pipeline.predict(X_train_fold)
            y_train_prob = pipeline.predict_proba(X_train_fold)[:, 1] if hasattr(pipeline, "predict_proba") else None
            y_test_pred = pipeline.predict(X_test_fold)
            y_test_prob = pipeline.predict_proba(X_test_fold)[:, 1] if hasattr(pipeline, "predict_proba") else None
            
            # เก็บ Metrics
            train_metrics = calculate_metrics(y_train, y_train_pred, y_train_prob, prefix="train_")
            test_metrics = calculate_metrics(y_test, y_test_pred, y_test_prob, prefix="")
            
            combined_metrics = {**train_metrics, **test_metrics}
            final_results[name]["metrics_per_fold"][f"fold_{fold}"] = combined_metrics
            fold_metrics.append(combined_metrics)
            
        # สร้างกราฟรวม (หลังรันครบ 5 Fold)
        if selection_method is not None and len(model_selection_histories) == 5:
            x_label = 'Number of Features Removed' if selection_method == 'backward' else 'Number of Features Added'
            
            # 1. กราฟ Subplot 5 ช่อง
            fig, axes = plt.subplots(1, 5, figsize=(25, 5), sharey=True)
            fig.suptitle(f'{selection_method.capitalize()} Selection History - {name}', fontsize=16)
            
            for fold, ax in zip(range(1, 6), axes):
                h_df = model_selection_histories[fold]
                ax.plot(h_df['Step'], h_df['AUC'], marker='o', label='AUC')
                ax.plot(h_df['Step'], h_df['F1'], marker='s', label='F1')
                ax.plot(h_df['Step'], h_df['Precision'], marker='^', label='Precision')
                
                ax.set_title(f'Fold {fold}')
                ax.set_xlabel(x_label)
                ax.grid(True)
                ax.set_ylim([0.0, 1.05])
                
            axes[0].set_ylabel('Score')
            axes[-1].legend(loc='lower left', bbox_to_anchor=(1.05, 0.5))
            
            plt.tight_layout()
            plot_combined_path = os.path.join(result_save_dir, f'{exp_name}_combined_metrics_plot.png')
            plt.savefig(plot_combined_path, bbox_inches='tight')
            plt.close()
            
            # 2. กราฟเปรียบเทียบ AUC
            plt.figure(figsize=(10, 6))
            for fold in range(1, 6):
                h_df = model_selection_histories[fold]
                plt.plot(h_df['Step'], h_df['AUC'], marker='o', label=f'Fold {fold} AUC')
                
            plt.title(f'AUC Comparison Across All 5 Folds - {name} ({selection_method.capitalize()})')
            plt.xlabel(x_label)
            plt.ylabel('AUC Score')
            plt.grid(True)
            plt.ylim([0.0, 1.05])
            plt.legend()
            
            plot_auc_compare_path = os.path.join(result_save_dir, f'{exp_name}_AUC_comparison_all_folds.png')
            plt.savefig(plot_auc_compare_path)
            plt.close()

        # คำนวณค่าเฉลี่ย
        avg_metrics = {}
        for key in fold_metrics[0].keys():
            if fold_metrics[0][key] is not None:
                if key in ["train_TN", "train_FP", "train_FN", "train_TP", "TN", "FP", "FN", "TP"]:
                    avg_metrics[key] = int(np.mean([m[key] for m in fold_metrics]))
                else:
                    avg_metrics[key] = np.mean([m[key] for m in fold_metrics])
        
        final_results[name]["confusionMatrix"] = avg_metrics
        
    return final_results

# 5. สั่งรัน
print("=== Starting Experiment: Full Feature Set ===")
full_features_results = evaluate_pre_split_folds(data_folder='../data', selection_method=None, exp_name="FullFeatures")
with open('saved_results/results_full_features.json', 'w', encoding='utf-8') as f:
    json.dump(full_features_results, f, ensure_ascii=False, indent=4)

print("\n=== Starting Experiment: Selected Feature Subset (Backward Selection) ===")
backward_features_results = evaluate_pre_split_folds(data_folder='../data', selection_method='backward', exp_name="BackwardFeatures")
with open('saved_results/results_backward_features.json', 'w', encoding='utf-8') as f:
    json.dump(backward_features_results, f, ensure_ascii=False, indent=4)

print("\n=== Starting Experiment: Selected Feature Subset (Forward Selection) ===")
forward_features_results = evaluate_pre_split_folds(data_folder='../data', selection_method='forward', exp_name="ForwardFeatures")
with open('saved_results/results_forward_features.json', 'w', encoding='utf-8') as f:
    json.dump(forward_features_results, f, ensure_ascii=False, indent=4)

print("\nAll tasks completed! Metrics are ready for comparison.")

=== Starting Experiment: Full Feature Set ===

--- Processing Fold 1 | Model: Logistic Regression | Method: None ---

--- Processing Fold 2 | Model: Logistic Regression | Method: None ---

--- Processing Fold 3 | Model: Logistic Regression | Method: None ---

--- Processing Fold 4 | Model: Logistic Regression | Method: None ---

--- Processing Fold 5 | Model: Logistic Regression | Method: None ---

--- Processing Fold 1 | Model: Naive Bayes | Method: None ---

--- Processing Fold 2 | Model: Naive Bayes | Method: None ---

--- Processing Fold 3 | Model: Naive Bayes | Method: None ---

--- Processing Fold 4 | Model: Naive Bayes | Method: None ---

--- Processing Fold 5 | Model: Naive Bayes | Method: None ---

--- Processing Fold 1 | Model: AdaBoost | Method: None ---

--- Processing Fold 2 | Model: AdaBoost | Method: None ---

--- Processing Fold 3 | Model: AdaBoost | Method: None ---

--- Processing Fold 4 | Model: AdaBoost | Method: None ---

--- Processing Fold 5 | Model: AdaBoost | M